# OFEV nested zip reader (station 2170)

This notebook scans the full nested archive recursively, extracts all PQPREVI files for station **2170**, and keeps only deterministic forecasts (skips ensemble/quantile tables).

It ignores no-rain files, any file with `ctrl` in its name, and any model with a `CON` token (e.g. `COSMOE_CON`), and produces:
- `df_det` (lead times 0–48h)
- `df_lt_0_6`, `df_lt_6_12`, `df_lt_12_24`, `df_lt_24_36`, `df_lt_36_48`

No CSV export.

In [1]:
from __future__ import annotations

import re
import zipfile
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd

ZIP_PATH = Path(r"..\data\Data Fornisseurs\OFEV_model\swisstransfer_e4eb4e12-cd87-4d66-be08-4bd5ea69dadf.zip")

STATION_ID = "2170"

# Parse all lead times up to 48h (we'll slice into windows after parsing)
LEAD_TIMES = set(range(49))

ROW_RE = re.compile(
    r"^\s*(\d{1,2})\s+(\d{1,2})\s+(\d{4})\s+(\d{2})\s+([\d.,]+)\s+([\d.,]+)\s*$"
)
NUM_RE = re.compile(r"^[+-]?(?:\d+(?:[\.,]\d*)?|[\.,]\d+)$")


def _f(x) -> float:
    try:
        s = str(x).strip().replace(" ", "").replace(",", ".")
        return float(s)
    except Exception:
        return np.nan


def iter_nested_files(zf: zipfile.ZipFile, prefix: str = ""):
    """Yield (virtual_path, raw_bytes) for all files in arbitrarily nested zip trees."""
    for info in zf.infolist():
        if info.is_dir():
            continue
        name = info.filename
        raw = zf.read(name)
        vpath = f"{prefix}{name}"

        if name.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(BytesIO(raw)) as nested:
                    yield from iter_nested_files(nested, prefix=f"{vpath}::")
            except zipfile.BadZipFile:
                yield vpath, raw
        else:
            yield vpath, raw


def parse_issue_time(vpath: str, text: str) -> pd.Timestamp:
    """Infer forecast issue time from the surrounding folder/zip name or file header."""
    # Common pattern in archive: yyyymmddhh_pqprevi (e.g., 2020121707_pqprevi.zip)
    m = re.search(r"(20\d{6})(\d{2})(?=\D)", vpath)
    if m:
        ymd, hh = m.group(1), m.group(2)
        return pd.Timestamp(f"{ymd[:4]}-{ymd[4:6]}-{ymd[6:8]} {hh}:00:00")

    # Newer pattern in archive (2022-12+ / 2023+): yyyymmdd-HHMM_..._pqprevi_... (e.g., 20240101-0730_Rhone_pqprevi_official.zip)
    m = re.search(r"(20\d{6})-(\d{2})(\d{2})(?=\D)", vpath)
    if m:
        ymd, hh = m.group(1), m.group(2)
        return pd.Timestamp(f"{ymd[:4]}-{ymd[4:6]}-{ymd[6:8]} {hh}:00:00")

    # Fallback: try to infer from header text
    m = re.search(r"(\d{2})\.(\d{2})\.(20\d{2}).{0,50}?(\d{2})h", text)
    if m:
        dd, mm, yyyy, hh = m.groups()
        return pd.Timestamp(f"{yyyy}-{mm}-{dd} {hh}:00:00")

    return pd.NaT


def model_from_pqprevi_filename(fname: str) -> str | None:
    """Parse model name from e.g. Pqprevi_C1E_2170.txt or Pqprevi_ICH2_Med_2170.txt."""
    base = Path(fname).name
    if not re.match(r"^pqprevi_", base, flags=re.I):
        return None
    m = re.match(r"^Pqprevi_(.+)_\d+\.txt$", base, flags=re.I)
    if not m:
        return None
    return m.group(1).upper()


def _pick(cols: list[str], *keys: str) -> str | None:
    cl = {c.lower(): c for c in cols}
    for k in keys:
        for lk, c in cl.items():
            if k in lk:
                return c
    return None


def read_tables(vpath: str, text: str, model: str, issue: pd.Timestamp) -> list[dict]:
    """Parse deterministic rows from one PQPREVI station file."""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        return []

    i = next((k for k, ln in enumerate(lines) if re.search(r"\bdd\s+mm\s+yyyy\s+hh\b", ln, re.I)), None)
    if i is None:
        rows: list[dict] = []
        for ln in lines:
            m = ROW_RE.match(ln)
            if not m:
                continue
            dd, mm, yyyy, hh, h, q = m.groups()
            vt = pd.Timestamp(f"{yyyy}-{mm}-{dd} {hh}:00:00")
            lt = int(round((vt - issue).total_seconds() / 3600))
            if lt in LEAD_TIMES and lt >= 0:
                rows.append({
                    "model": model,
                    "issue_time": issue,
                    "valid_time": vt,
                    "lead_time_h": lt,
                    "H": _f(h),
                    "Q": _f(q),
                    "source_path": vpath,
                })
        return rows

    head = re.split(r"\s+", lines[i])
    if len(head) < 6:
        return []

    base, cols = ["dd", "mm", "yyyy", "hh"], head[4:]

    # Skip ensemble/quantile-style tables entirely (deterministic-only notebook)
    is_ensemble = any(re.fullmatch(r"[HQ]_(ctl|e\d{2}|min|p25|p50|p75|max)", c, flags=re.I) for c in cols)
    if is_ensemble:
        return []

    h_col = _pick(cols, "wasserstand", "niveau", "h")
    q_col = _pick(cols, "abfluss", "debit", "dbit", "débit", "débits", "q")
    if not h_col and not q_col:
        return []

    det: list[dict] = []
    for ln in lines[i + 1 :]:
        toks = re.split(r"\s+", ln)
        if len(toks) < 4 + len(cols) or not all(NUM_RE.match(t) for t in toks[:4]):
            continue
        row = dict(zip(base + cols, toks[: 4 + len(cols)]))
        vt = pd.Timestamp(f"{row['yyyy']}-{row['mm']}-{row['dd']} {row['hh']}:00:00")
        lt = int(round((vt - issue).total_seconds() / 3600))
        if lt not in LEAD_TIMES or lt < 0:
            continue

        det.append({
            "model": model,
            "issue_time": issue,
            "valid_time": vt,
            "lead_time_h": lt,
            "H": _f(row.get(h_col, np.nan)) if h_col else np.nan,
            "Q": _f(row.get(q_col, np.nan)) if q_col else np.nan,
            "source_path": vpath,
        })

    return det


def _df(rows: list[dict], cols: list[str]) -> pd.DataFrame:
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=cols)


det_rows: list[dict] = []
with zipfile.ZipFile(ZIP_PATH) as zf:
    for vpath, raw in iter_nested_files(zf):
        fname = Path(vpath.split("::")[-1]).name
        lower_fname = fname.lower()

        if not (lower_fname.startswith("pqprevi_") and lower_fname.endswith(f"_{STATION_ID}.txt")):
            continue
        if "norain" in lower_fname:
            continue
        if re.search(r"(?:^|_)ctrl(?:_|\.|$)", lower_fname):
            continue

        model = model_from_pqprevi_filename(fname)
        if not model:
            continue
        # Skip models with a 'CON' token (e.g., COSMOE_CON)
        if any(part == 'CON' for part in model.split('_')):
            continue

        text = raw.decode("utf-8", errors="ignore")
        issue = parse_issue_time(vpath, text)
        if pd.isna(issue):
            continue

        det_rows.extend(read_tables(vpath, text, model, issue))

if not det_rows:
    raise RuntimeError(f"No station {STATION_ID} deterministic data found")

# Deterministic DF

df_det = (
    _df(det_rows, ["model", "issue_time", "valid_time", "lead_time_h", "H", "Q", "source_path"])
    .sort_values(["lead_time_h", "issue_time", "model"], na_position="last")
    .reset_index(drop=True)
)

# Lead-time windows (no CSV export)
# Convention: first window includes 6, next windows are (6, 12], (12, 24], (24, 36], (36, 48]

lt = df_det["lead_time_h"]

df_lt_0_6 = df_det[lt.between(0, 6)].reset_index(drop=True)
df_lt_6_12 = df_det[(lt > 6) & (lt <= 12)].reset_index(drop=True)
df_lt_12_24 = df_det[(lt > 12) & (lt <= 24)].reset_index(drop=True)
df_lt_24_36 = df_det[(lt > 24) & (lt <= 36)].reset_index(drop=True)
df_lt_36_48 = df_det[(lt > 36) & (lt <= 48)].reset_index(drop=True)

print("deterministic rows (all lead times 0-48):", len(df_det))
print("rows  0- 6:", len(df_lt_0_6))
print("rows  6-12:", len(df_lt_6_12))
print("rows 12-24:", len(df_lt_12_24))
print("rows 24-36:", len(df_lt_24_36))
print("rows 36-48:", len(df_lt_36_48))

deterministic rows (all lead times 0-48): 323791
rows  0- 6: 53735
rows  6-12: 46054
rows 12-24: 91973
rows 24-36: 69642
rows 36-48: 62387


In [2]:
display(df_lt_0_6.head())
display(df_lt_6_12.head())
display(df_lt_12_24.head())
display(df_lt_24_36.head())
display(df_lt_36_48.head())

,model,issue_time,valid_time,lead_time_h,H,Q,source_path
0,COSMO1,2020-06-11 07:00:00,2020-06-11 07:00:00,0,379.92,111.8,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
1,COSMO7,2020-06-11 07:00:00,2020-06-11 07:00:00,0,379.92,111.8,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
2,COSMOE_MED,2020-06-11 07:00:00,2020-06-11 07:00:00,0,379.92,111.8,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
3,ECMNOR,2020-06-11 07:00:00,2020-06-11 07:00:00,0,379.92,111.8,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
4,ECMWF,2020-06-11 07:00:00,2020-06-11 07:00:00,0,379.92,111.8,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...


,model,issue_time,valid_time,lead_time_h,H,Q,source_path
0,COSMO1,2020-06-11 07:00:00,2020-06-11 14:00:00,7,379.92,111.9,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
1,COSMO7,2020-06-11 07:00:00,2020-06-11 14:00:00,7,379.93,112.0,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
2,COSMOE_MED,2020-06-11 07:00:00,2020-06-11 14:00:00,7,379.92,111.9,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
3,ECMNOR,2020-06-11 07:00:00,2020-06-11 14:00:00,7,379.93,112.0,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
4,ECMWF,2020-06-11 07:00:00,2020-06-11 14:00:00,7,379.93,112.2,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...


,model,issue_time,valid_time,lead_time_h,H,Q,source_path
0,COSMO1,2020-06-11 07:00:00,2020-06-11 20:00:00,13,379.85,102.2,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
1,COSMO7,2020-06-11 07:00:00,2020-06-11 20:00:00,13,379.89,107.2,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
2,COSMOE_MED,2020-06-11 07:00:00,2020-06-11 20:00:00,13,379.85,102.2,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
3,ECMNOR,2020-06-11 07:00:00,2020-06-11 20:00:00,13,379.85,102.1,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
4,ECMWF,2020-06-11 07:00:00,2020-06-11 20:00:00,13,379.88,106.0,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...


,model,issue_time,valid_time,lead_time_h,H,Q,source_path
0,COSMO1,2020-06-11 07:00:00,2020-06-12 08:00:00,25,379.94,114.2,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
1,COSMO7,2020-06-11 07:00:00,2020-06-12 08:00:00,25,379.99,120.8,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
2,COSMOE_MED,2020-06-11 07:00:00,2020-06-12 08:00:00,25,379.83,99.9,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
3,ECMNOR,2020-06-11 07:00:00,2020-06-12 08:00:00,25,379.79,94.8,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
4,ECMWF,2020-06-11 07:00:00,2020-06-12 08:00:00,25,379.87,105.3,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...


,model,issue_time,valid_time,lead_time_h,H,Q,source_path
0,COSMO7,2020-06-11 07:00:00,2020-06-12 20:00:00,37,379.80,95.7,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
1,COSMOE_MED,2020-06-11 07:00:00,2020-06-12 20:00:00,37,379.74,88.6,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
2,ECMNOR,2020-06-11 07:00:00,2020-06-12 20:00:00,37,379.70,83.5,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
3,ECMWF,2020-06-11 07:00:00,2020-06-12 20:00:00,37,379.75,89.7,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
4,COSMO7,2020-06-12 08:00:00,2020-06-13 21:00:00,37,379.77,91.6,2020.zip::2020/2020061208_pqprevi.zip::Pqprevi...


In [3]:
# Sanity check: make sure we now cover years beyond 2022
print("issue_time min:", df_det["issue_time"].min())
print("issue_time max:", df_det["issue_time"].max())
print("rows by issue year:")
print(df_det.assign(year=df_det["issue_time"].dt.year).groupby("year").size())


issue_time min: 2020-06-11 07:00:00
issue_time max: 2025-12-31 07:00:00
rows by issue year:
year
2020    43815
2021    70637
2022    56696
2023    60416
2024    46310
2025    45917
dtype: int64


In [4]:
# Check which models remain (should not include '*_CON')
models = sorted(df_det["model"].dropna().unique())
print("n models:", len(models))
print(models)
print("any CON token models present?", any("CON" in m.split("_") for m in models))


n models: 9
['C1E_MED', 'C2E_MED', 'COSMO1', 'COSMO7', 'COSMOE_MED', 'ECMNOR', 'ECMWF', 'ICH1_MED', 'ICH2_MED']
any CON token models present? False


In [5]:
def _sort_and_median_by_issue_lead(df: pd.DataFrame, *, name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    # 1) sort rows (issue_time, then lead_time within each issue)
    df_sorted = (
        df.sort_values(["issue_time", "lead_time_h", "model"], na_position="last")
          .reset_index(drop=True)
    )

    # 2) median across models for each (issue_time, lead_time_h)
    # valid_time should be identical across models for same (issue, lead) so we keep the first.
    df_med = (
        df_sorted
        .groupby(["issue_time", "lead_time_h"], as_index=False)
        .agg(
            valid_time=("valid_time", "first"),
            H_median=("H", "median"),
            Q_median=("Q", "median"),
            n_models=("model", "nunique"),
        )
        .sort_values(["issue_time", "lead_time_h"])
        .reset_index(drop=True)
    )

    print(f"{name}: sorted rows={len(df_sorted):,} | median rows={len(df_med):,}")
    return df_sorted, df_med


# Apply to all your dataframes
df_det_sorted, df_det_median = _sort_and_median_by_issue_lead(df_det, name="df_det")
df_lt_0_6_sorted, df_lt_0_6_median = _sort_and_median_by_issue_lead(df_lt_0_6, name="df_lt_0_6")
df_lt_6_12_sorted, df_lt_6_12_median = _sort_and_median_by_issue_lead(df_lt_6_12, name="df_lt_6_12")
df_lt_12_24_sorted, df_lt_12_24_median = _sort_and_median_by_issue_lead(df_lt_12_24, name="df_lt_12_24")
df_lt_24_36_sorted, df_lt_24_36_median = _sort_and_median_by_issue_lead(df_lt_24_36, name="df_lt_24_36")
df_lt_36_48_sorted, df_lt_36_48_median = _sort_and_median_by_issue_lead(df_lt_36_48, name="df_lt_36_48")

df_det: sorted rows=323,791 | median rows=120,589
df_lt_0_6: sorted rows=53,735 | median rows=17,227
df_lt_6_12: sorted rows=46,054 | median rows=14,766
df_lt_12_24: sorted rows=91,973 | median rows=29,532
df_lt_24_36: sorted rows=69,642 | median rows=29,532
df_lt_36_48: sorted rows=62,387 | median rows=29,532


In [7]:
from pathlib import Path

OUT_DIR = Path("..") / "outputs" / "OFEV_probabilistic"
OUT_DIR.mkdir(parents=True, exist_ok=True)

dfs_to_save = {
    "station2170_det_median_lt0_48.csv": df_det_median,
    "station2170_det_median_lt0_6.csv": df_lt_0_6_median,
    "station2170_det_median_lt6_12.csv": df_lt_6_12_median,
    "station2170_det_median_lt12_24.csv": df_lt_12_24_median,
    "station2170_det_median_lt24_36.csv": df_lt_24_36_median,
    "station2170_det_median_lt36_48.csv": df_lt_36_48_median,
}

for fname, df in dfs_to_save.items():
    out_path = OUT_DIR / fname
    df.to_csv(out_path, index=False)
    print("wrote:", out_path, "| rows:", len(df))

wrote: ..\outputs\OFEV_probabilistic\station2170_det_median_lt0_48.csv | rows: 120589
wrote: ..\outputs\OFEV_probabilistic\station2170_det_median_lt0_6.csv | rows: 17227
wrote: ..\outputs\OFEV_probabilistic\station2170_det_median_lt6_12.csv | rows: 14766
wrote: ..\outputs\OFEV_probabilistic\station2170_det_median_lt12_24.csv | rows: 29532
wrote: ..\outputs\OFEV_probabilistic\station2170_det_median_lt24_36.csv | rows: 29532
wrote: ..\outputs\OFEV_probabilistic\station2170_det_median_lt36_48.csv | rows: 29532
